# ML Methods - Direction Prediction

# Imports

In [1]:
from pathlib import Path
import sys
import gc
import os
import random
import warnings

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "exploration" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from technical.labelling import label_direction_future_avg

import xgboost as xgb
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(1)


# Data Loading

In [2]:
SYMBOL = "AAPL"
DATA_DIR = PROJECT_ROOT / "data"
BOOK_PATH = DATA_DIR / f"{SYMBOL}_1s.parquet"
EVENT_PATH = DATA_DIR / f"{SYMBOL}_evt.parquet"

TICK_SIZE = 0.01
HORIZON = 20
SESSION_GAP_SECONDS = 3600
EVENT_FEATURE_LAG_SECONDS = 1
ROLLING_WARMUP = 20
TEST_SESSION_COUNT = 5

book = pd.read_parquet(BOOK_PATH).sort_values("time_abs_s").reset_index(drop=True)
book["time_abs_s"] = book["time_abs_s"].astype(np.int64)

event_cols = ["time_abs_s", "event_type", "size", "direction"]
events = pd.read_parquet(
    EVENT_PATH,
    columns=event_cols,
    filters=[("event_type", "in", [1, 2, 3, 4, 5])],
)
events = events.sort_values("time_abs_s").reset_index(drop=True)
events["time_abs_s"] = events["time_abs_s"].astype(np.int64)
events["event_type"] = events["event_type"].astype(np.int8)
events["direction"] = events["direction"].astype(np.int8)
events["size"] = events["size"].astype(np.float32)


# Feature Engineering

In [3]:
def add_sessions(df):
    out = df.sort_values("time_abs_s").reset_index(drop=True).copy()
    dt = out["time_abs_s"].diff()
    is_new_session = (dt < 0) | (dt > SESSION_GAP_SECONDS)
    out["session_id"] = is_new_session.fillna(False).cumsum().astype(int)
    out["session_pos"] = out.groupby("session_id").cumcount().astype(int)
    out["session_len"] = out.groupby("session_id")["time_abs_s"].transform("size").astype(int)
    out["session_progress"] = (out["session_pos"] / (out["session_len"] - 1)).fillna(0.0).astype(np.float32)
    return out

book = add_sessions(book)

In [4]:
def aggregate_message_features(events):
    evt = events.copy()
    event_type = evt["event_type"].to_numpy(dtype=np.int8)
    direction = evt["direction"].to_numpy(dtype=np.int8)
    size = evt["size"].to_numpy(dtype=np.float32)

    add_mask = event_type == 1
    cancel_mask = np.isin(event_type, [2, 3])
    visible_exec_mask = event_type == 4
    hidden_exec_mask = event_type == 5
    trade_mask = visible_exec_mask | hidden_exec_mask

    evt["limit_add_signed_1s"] = np.where(add_mask, direction * size, 0.0).astype(np.float32)
    evt["limit_add_abs_1s"] = np.where(add_mask, size, 0.0).astype(np.float32)

    evt["cancel_pressure_signed_1s"] = np.where(cancel_mask, -direction * size, 0.0).astype(np.float32)
    evt["cancel_abs_1s"] = np.where(cancel_mask, size, 0.0).astype(np.float32)

    evt["aggr_trade_signed_1s"] = np.where(visible_exec_mask, -direction * size, 0.0).astype(np.float32)
    evt["aggr_trade_abs_1s"] = np.where(visible_exec_mask, size, 0.0).astype(np.float32)

    evt["hidden_trade_signed_1s"] = np.where(hidden_exec_mask, -direction * size, 0.0).astype(np.float32)
    evt["hidden_trade_abs_1s"] = np.where(hidden_exec_mask, size, 0.0).astype(np.float32)

    evt["event_count_1s"] = 1
    evt["trade_count_1s"] = np.where(trade_mask, 1, 0).astype(np.int16)

    message_features = (
        evt.groupby("time_abs_s", sort=True)
        .agg(
            limit_add_signed_1s=("limit_add_signed_1s", "sum"),
            limit_add_abs_1s=("limit_add_abs_1s", "sum"),
            cancel_pressure_signed_1s=("cancel_pressure_signed_1s", "sum"),
            cancel_abs_1s=("cancel_abs_1s", "sum"),
            aggr_trade_signed_1s=("aggr_trade_signed_1s", "sum"),
            aggr_trade_abs_1s=("aggr_trade_abs_1s", "sum"),
            hidden_trade_signed_1s=("hidden_trade_signed_1s", "sum"),
            hidden_trade_abs_1s=("hidden_trade_abs_1s", "sum"),
            event_count_1s=("event_count_1s", "sum"),
            trade_count_1s=("trade_count_1s", "sum"),
        )
        .reset_index()
    )
    message_features["time_abs_s"] = message_features["time_abs_s"] + EVENT_FEATURE_LAG_SECONDS
    return message_features

message_1s = aggregate_message_features(events)
del events
gc.collect()

book = book.merge(message_1s, on="time_abs_s", how="left")
message_cols = [
    "limit_add_signed_1s", "limit_add_abs_1s",
    "cancel_pressure_signed_1s", "cancel_abs_1s",
    "aggr_trade_signed_1s", "aggr_trade_abs_1s",
    "hidden_trade_signed_1s", "hidden_trade_abs_1s",
    "event_count_1s", "trade_count_1s",
]
book[message_cols] = book[message_cols].fillna(0.0)
book["event_count_1s"] = book["event_count_1s"].astype(np.int32)
book["trade_count_1s"] = book["trade_count_1s"].astype(np.int32)
for col in set(message_cols) - {"event_count_1s", "trade_count_1s"}:
    book[col] = book[col].astype(np.float32)


In [5]:
selected_features = [
    "imb_l1",
    "cum_imb_l3",
    "cum_imb_l5",
    "cum_imb_l10",
    "imbalance_gradient_l3_l10",
    "bid_depth_l3",
    "ask_depth_l3",
    "depth_ratio_l3",
    "spread_ticks",
    "one_tick_spread",
    "touch_depth",
    "cum_depth_l3",
    "cum_depth_l5",
    "cum_depth_l10",
    "microprice",
    "microbias_ticks",
    "microbias_over_spread",
    "event_count_3",
    "event_count_10",
    "event_count_20",
    "trade_count_3",
    "trade_count_10",
    "trade_count_20",
    "ofi_l1_norm_w3",
    "ofi_l1_norm_w10",
    "ofi_l1_norm_w20",
    "ofi_change_3",
    "ofi_change_5",
    "ofi_change_20",
    "mid_ret_3",
    "mid_ret_10",
    "mid_ret_20",
    "realized_vol_10",
    "realized_vol_20",
    "return_abs_5",
    "sma_10_dist",
    "sma_20_dist",
    "rsi_14",
    "macd",
    "rolling_return_10",
    "rolling_return_20",
    "bb_position_20",
    "bb_width_20",
    "vwap",
]


In [6]:
def rolling_sum_by_session(series, session_id, window):
    if window == 1:
        return series.astype(np.float32)
    return (
        series.groupby(session_id, sort=False)
        .rolling(window, min_periods=window)
        .sum()
        .reset_index(level=0, drop=True)
        .fillna(0.0)
        .astype(np.float32)
    )


def rolling_mean_by_session(series, session_id, window):
    return (
        series.groupby(session_id, sort=False)
        .rolling(window, min_periods=window)
        .mean()
        .reset_index(level=0, drop=True)
        .astype(np.float32)
    )


def rolling_std_by_session(series, session_id, window):
    return (
        series.groupby(session_id, sort=False)
        .rolling(window, min_periods=window)
        .std()
        .reset_index(level=0, drop=True)
        .astype(np.float32)
    )


def ewm_mean_by_session(series, session_id, span):
    return (
        series.groupby(session_id, sort=False)
        .ewm(span=span, adjust=False)
        .mean()
        .reset_index(level=0, drop=True)
        .astype(np.float32)
    )


def compute_ofi_level(df, level):
    grouped = df.groupby("session_id", sort=False)
    bid = df[f"bid{level}"].astype(float)
    ask = df[f"ask{level}"].astype(float)
    bid_size = df[f"bid{level}_sz"].astype(float)
    ask_size = df[f"ask{level}_sz"].astype(float)

    previous_bid = grouped[f"bid{level}"].shift(1)
    previous_ask = grouped[f"ask{level}"].shift(1)
    previous_bid_size = grouped[f"bid{level}_sz"].shift(1).fillna(0.0)
    previous_ask_size = grouped[f"ask{level}_sz"].shift(1).fillna(0.0)

    bid_change = bid - previous_bid
    ask_change = ask - previous_ask
    bid_size_change = bid_size - previous_bid_size
    ask_size_change = ask_size - previous_ask_size

    bid_ofi = np.where(bid_change > 0, bid_size, np.where(bid_change < 0, -previous_bid_size, bid_size_change))
    ask_ofi = np.where(ask_change < 0, -ask_size, np.where(ask_change > 0, previous_ask_size, -ask_size_change))
    return pd.Series(bid_ofi + ask_ofi, index=df.index).fillna(0.0).astype(np.float32)


def build_selected_features(df):
    out = df.copy()
    eps = 1e-9

    book_numeric_cols = [
        "mid",
        *[f"bid{i}" for i in range(1, 11)],
        *[f"ask{i}" for i in range(1, 11)],
        *[f"bid{i}_sz" for i in range(1, 11)],
        *[f"ask{i}_sz" for i in range(1, 11)],
    ]
    out[book_numeric_cols] = out[book_numeric_cols].apply(pd.to_numeric, errors="coerce").astype(np.float32)
    out["mid"] = out["mid"].fillna((out["bid1"] + out["ask1"]) * 0.5).astype(np.float32)

    grouped = out.groupby("session_id", sort=False)

    out["spread"] = (out["ask1"] - out["bid1"]).astype(np.float32)
    out["spread_ticks"] = (out["spread"] / TICK_SIZE).astype(np.float32)
    out["one_tick_spread"] = (out["spread_ticks"] <= 1.01).astype(np.float32)

    out["touch_depth"] = (out["bid1_sz"] + out["ask1_sz"]).astype(np.float32)
    out["imb_l1"] = ((out["bid1_sz"] - out["ask1_sz"]) / (out["touch_depth"] + eps)).astype(np.float32)

    out["bid_depth_l3"] = out[["bid1_sz", "bid2_sz", "bid3_sz"]].sum(axis=1).astype(np.float32)
    out["ask_depth_l3"] = out[["ask1_sz", "ask2_sz", "ask3_sz"]].sum(axis=1).astype(np.float32)
    bid_depth_l5 = out[[f"bid{i}_sz" for i in range(1, 6)]].sum(axis=1).astype(np.float32)
    ask_depth_l5 = out[[f"ask{i}_sz" for i in range(1, 6)]].sum(axis=1).astype(np.float32)
    bid_depth_l10 = out[[f"bid{i}_sz" for i in range(1, 11)]].sum(axis=1).astype(np.float32)
    ask_depth_l10 = out[[f"ask{i}_sz" for i in range(1, 11)]].sum(axis=1).astype(np.float32)

    out["cum_depth_l3"] = (out["bid_depth_l3"] + out["ask_depth_l3"]).astype(np.float32)
    out["cum_depth_l5"] = (bid_depth_l5 + ask_depth_l5).astype(np.float32)
    out["cum_depth_l10"] = (bid_depth_l10 + ask_depth_l10).astype(np.float32)

    out["cum_imb_l3"] = ((out["bid_depth_l3"] - out["ask_depth_l3"]) / (out["cum_depth_l3"] + eps)).astype(np.float32)
    out["cum_imb_l5"] = ((bid_depth_l5 - ask_depth_l5) / (out["cum_depth_l5"] + eps)).astype(np.float32)
    out["cum_imb_l10"] = ((bid_depth_l10 - ask_depth_l10) / (out["cum_depth_l10"] + eps)).astype(np.float32)
    out["imbalance_gradient_l3_l10"] = (out["cum_imb_l3"] - out["cum_imb_l10"]).astype(np.float32)
    out["depth_ratio_l3"] = (out["bid_depth_l3"] / (out["ask_depth_l3"] + eps)).astype(np.float32)

    out["microprice"] = (
        (out["bid1"] * out["ask1_sz"] + out["ask1"] * out["bid1_sz"])
        / (out["touch_depth"] + eps)
    ).astype(np.float32)
    out["microbias_ticks"] = ((out["microprice"] - out["mid"]) / TICK_SIZE).astype(np.float32)
    out["microbias_over_spread"] = (
        (out["microprice"] - out["mid"]) / (out["spread"] + eps)
    ).replace([np.inf, -np.inf], 0.0).fillna(0.0).astype(np.float32)

    out["event_count_1"] = out["event_count_1s"].astype(np.float32)
    out["trade_count_1"] = out["trade_count_1s"].astype(np.float32)
    for window in [3, 10, 20]:
        out[f"event_count_{window}"] = rolling_sum_by_session(out["event_count_1"], out["session_id"], window)
        out[f"trade_count_{window}"] = rolling_sum_by_session(out["trade_count_1"], out["session_id"], window)

    out["ofi_l1_raw"] = compute_ofi_level(out, 1)
    for window in [3, 10, 20]:
        ofi_sum = rolling_sum_by_session(out["ofi_l1_raw"], out["session_id"], window)
        out[f"ofi_l1_norm_w{window}"] = (
            ofi_sum / (out["touch_depth"] + eps)
        ).replace([np.inf, -np.inf], 0.0).fillna(0.0).astype(np.float32)

    for window in [3, 5, 20]:
        previous_ofi = grouped["ofi_l1_raw"].shift(window).fillna(0.0)
        out[f"ofi_change_{window}"] = (
            (out["ofi_l1_raw"] - previous_ofi) / (out["touch_depth"] + eps)
        ).replace([np.inf, -np.inf], 0.0).fillna(0.0).astype(np.float32)

    log_mid = np.log(out["mid"].astype(float).clip(lower=eps))
    out["mid_ret_1"] = (
        log_mid.groupby(out["session_id"])
        .diff()
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .astype(np.float32)
    )
    for window in [3, 10, 20]:
        out[f"mid_ret_{window}"] = rolling_sum_by_session(out["mid_ret_1"], out["session_id"], window)
        out[f"rolling_return_{window}"] = out[f"mid_ret_{window}"].astype(np.float32)

    out["realized_vol_10"] = np.sqrt(
        rolling_sum_by_session(out["mid_ret_1"].astype(float) ** 2, out["session_id"], 10)
    ).astype(np.float32)
    out["realized_vol_20"] = np.sqrt(
        rolling_sum_by_session(out["mid_ret_1"].astype(float) ** 2, out["session_id"], 20)
    ).astype(np.float32)
    out["return_abs_5"] = rolling_sum_by_session(out["mid_ret_1"].abs(), out["session_id"], 5)

    sma_10 = rolling_mean_by_session(out["mid"], out["session_id"], 10)
    sma_20 = rolling_mean_by_session(out["mid"], out["session_id"], 20)
    out["sma_10_dist"] = ((out["mid"] - sma_10) / (sma_10 + eps)).astype(np.float32)
    out["sma_20_dist"] = ((out["mid"] - sma_20) / (sma_20 + eps)).astype(np.float32)

    price_change = grouped["mid"].diff().fillna(0.0)
    gain = price_change.clip(lower=0.0)
    loss = (-price_change).clip(lower=0.0)
    avg_gain = rolling_mean_by_session(gain, out["session_id"], 14)
    avg_loss = rolling_mean_by_session(loss, out["session_id"], 14)
    rs = avg_gain / (avg_loss + eps)
    out["rsi_14"] = (100.0 - (100.0 / (1.0 + rs))).fillna(50.0).astype(np.float32)

    ema_12 = ewm_mean_by_session(out["mid"], out["session_id"], 12)
    ema_26 = ewm_mean_by_session(out["mid"], out["session_id"], 26)
    out["macd"] = (ema_12 - ema_26).astype(np.float32)

    bb_mean_20 = sma_20
    bb_std_20 = rolling_std_by_session(out["mid"], out["session_id"], 20)
    bb_upper_20 = bb_mean_20 + 2.0 * bb_std_20
    bb_lower_20 = bb_mean_20 - 2.0 * bb_std_20
    out["bb_position_20"] = (
        (out["mid"] - bb_lower_20) / (bb_upper_20 - bb_lower_20 + eps)
    ).replace([np.inf, -np.inf], 0.5).fillna(0.5).astype(np.float32)
    out["bb_width_20"] = (
        (bb_upper_20 - bb_lower_20) / (out["mid"].abs() + eps)
    ).replace([np.inf, -np.inf], 0.0).fillna(0.0).astype(np.float32)

    trade_volume_1 = (out["aggr_trade_abs_1s"] + out["hidden_trade_abs_1s"]).astype(np.float32)
    vwap_num = rolling_sum_by_session(out["mid"] * trade_volume_1, out["session_id"], 20)
    vwap_den = rolling_sum_by_session(trade_volume_1, out["session_id"], 20)
    out["vwap"] = (vwap_num / vwap_den.replace(0.0, np.nan)).fillna(sma_20).fillna(out["mid"]).astype(np.float32)

    out[selected_features] = (
        out[selected_features]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .astype(np.float32)
    )
    return out

book = build_selected_features(book)


# Target Construction

In [7]:
def label_one_session(session_df):
    return label_direction_future_avg(
        session_df,
        price_col="mid",
        horizon=HORIZON,
        out_col="y_dir",
        avg_mode="mean",
        balanced=True,
        verbose=False,
    )

book = (
    book.groupby("session_id", group_keys=False, sort=False)
    .apply(label_one_session)
    .reset_index(drop=True)
)

model_df = book[book["session_pos"] >= ROLLING_WARMUP].copy().reset_index(drop=True)


# Train/Test Split

In [8]:
sessions = sorted(model_df["session_id"].unique())
test_sessions = sessions[-TEST_SESSION_COUNT:]
train_sessions = sessions[:-TEST_SESSION_COUNT]

model_df["split"] = np.where(model_df["session_id"].isin(test_sessions), "test", "train")

train_df = model_df[model_df["split"] == "train"].copy().reset_index(drop=True)
test_df = model_df[model_df["split"] == "test"].copy().reset_index(drop=True)

X_train = train_df[selected_features]
y_train = train_df["y_dir"]
X_test = test_df[selected_features]
y_test = test_df["y_dir"]

class_names = ["down", "up", "neutral"]
results = []


# Random Baseline

In [9]:
random_baseline_seed = 42


In [10]:
random_generator = np.random.default_rng(random_baseline_seed)
random_pred = random_generator.choice([0, 1, 2], size=len(y_test))

random_cm = pd.DataFrame(
    confusion_matrix(y_test, random_pred, labels=[0, 1, 2]),
    index=class_names,
    columns=class_names,
)
display(random_cm)

print(classification_report(y_test, random_pred, labels=[0, 1, 2], target_names=class_names, zero_division=0))

random_report = classification_report(y_test, random_pred, labels=[0, 1, 2], target_names=class_names, zero_division=0, output_dict=True)
results.append({
    "model": "Random Baseline",
    "accuracy": random_report["accuracy"],
    "macro_f1": random_report["macro avg"]["f1-score"],
    "weighted_f1": random_report["weighted avg"]["f1-score"],
})


,down,up,neutral
down,12951,12904,13040
up,13021,12948,12930
neutral,12960,12936,13044


              precision    recall  f1-score   support

        down       0.33      0.33      0.33     38895
          up       0.33      0.33      0.33     38899
     neutral       0.33      0.33      0.33     38940

    accuracy                           0.33    116734
   macro avg       0.33      0.33      0.33    116734
weighted avg       0.33      0.33      0.33    116734



# Logistic Regression

In [11]:
logistic_C = 1.0
logistic_max_iter = 1000


In [12]:
logistic_scaler = StandardScaler()
X_train_logistic = logistic_scaler.fit_transform(X_train)
X_test_logistic = logistic_scaler.transform(X_test)

logistic_model = LogisticRegression(
    C=logistic_C,
    max_iter=logistic_max_iter,
    solver="lbfgs",
    random_state=RANDOM_STATE,
)
logistic_model.fit(X_train_logistic, y_train)
logistic_pred = logistic_model.predict(X_test_logistic)

logistic_cm = pd.DataFrame(
    confusion_matrix(y_test, logistic_pred, labels=[0, 1, 2]),
    index=class_names,
    columns=class_names,
)
display(logistic_cm)

print(classification_report(y_test, logistic_pred, labels=[0, 1, 2], target_names=class_names, zero_division=0))

logistic_report = classification_report(y_test, logistic_pred, labels=[0, 1, 2], target_names=class_names, zero_division=0, output_dict=True)
results.append({
    "model": "Logistic Regression",
    "accuracy": logistic_report["accuracy"],
    "macro_f1": logistic_report["macro avg"]["f1-score"],
    "weighted_f1": logistic_report["weighted avg"]["f1-score"],
})


,down,up,neutral
down,10857,10449,17589
up,10609,11633,16657
neutral,8561,8239,22140


              precision    recall  f1-score   support

        down       0.36      0.28      0.32     38895
          up       0.38      0.30      0.34     38899
     neutral       0.39      0.57      0.46     38940

    accuracy                           0.38    116734
   macro avg       0.38      0.38      0.37    116734
weighted avg       0.38      0.38      0.37    116734



# Random Forest

In [13]:
rf_n_estimators = 200
rf_max_depth = 8
rf_min_samples_leaf = 20


In [14]:
rf_model = RandomForestClassifier(
    n_estimators=rf_n_estimators,
    max_depth=rf_max_depth,
    min_samples_leaf=rf_min_samples_leaf,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_cm = pd.DataFrame(
    confusion_matrix(y_test, rf_pred, labels=[0, 1, 2]),
    index=class_names,
    columns=class_names,
)
display(rf_cm)

print(classification_report(y_test, rf_pred, labels=[0, 1, 2], target_names=class_names, zero_division=0))

rf_report = classification_report(y_test, rf_pred, labels=[0, 1, 2], target_names=class_names, zero_division=0, output_dict=True)
results.append({
    "model": "Random Forest",
    "accuracy": rf_report["accuracy"],
    "macro_f1": rf_report["macro avg"]["f1-score"],
    "weighted_f1": rf_report["weighted avg"]["f1-score"],
})


,down,up,neutral
down,12104,13712,13079
up,11990,14719,12190
neutral,10074,11685,17181


              precision    recall  f1-score   support

        down       0.35      0.31      0.33     38895
          up       0.37      0.38      0.37     38899
     neutral       0.40      0.44      0.42     38940

    accuracy                           0.38    116734
   macro avg       0.38      0.38      0.38    116734
weighted avg       0.38      0.38      0.38    116734



# XGBoost

In [15]:
xgb_params = dict(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    tree_method="hist",
    n_estimators=400,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2.0,
    reg_alpha=0.0,
    gamma=0.0,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0,
)


In [16]:
xgb_model = xgb.XGBClassifier(**xgb_params)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

xgb_cm = pd.DataFrame(
    confusion_matrix(y_test, xgb_pred, labels=[0, 1, 2]),
    index=class_names,
    columns=class_names,
)
display(xgb_cm)

print(classification_report(y_test, xgb_pred, labels=[0, 1, 2], target_names=class_names, zero_division=0))

xgb_report = classification_report(y_test, xgb_pred, labels=[0, 1, 2], target_names=class_names, zero_division=0, output_dict=True)
results.append({
    "model": "XGBoost",
    "accuracy": xgb_report["accuracy"],
    "macro_f1": xgb_report["macro avg"]["f1-score"],
    "weighted_f1": xgb_report["weighted avg"]["f1-score"],
})


,down,up,neutral
down,14408,14200,10287
up,14126,15129,9644
neutral,12788,12702,13450


              precision    recall  f1-score   support

        down       0.35      0.37      0.36     38895
          up       0.36      0.39      0.37     38899
     neutral       0.40      0.35      0.37     38940

    accuracy                           0.37    116734
   macro avg       0.37      0.37      0.37    116734
weighted avg       0.37      0.37      0.37    116734



# LSTM

In [17]:
lstm_sequence_length = 20
lstm_hidden_size = 32
lstm_batch_size = 512
lstm_epochs = 50
lstm_patience = 5
lstm_learning_rate = 0.001


In [18]:
def make_lstm_sequences(df, X_values, y_values, sequence_length):
    X_sequences = []
    y_sequences = []
    session_values = df["session_id"].to_numpy()

    for session_id in df["session_id"].unique():
        rows = np.where(session_values == session_id)[0]
        session_X = X_values[rows]
        session_y = y_values[rows]

        for i in range(sequence_length - 1, len(session_X)):
            X_sequences.append(session_X[i - sequence_length + 1:i + 1])
            y_sequences.append(session_y[i])

    return np.array(X_sequences, dtype=np.float32), np.array(y_sequences, dtype=np.int64)

lstm_scaler = StandardScaler()
X_train_lstm_scaled = lstm_scaler.fit_transform(X_train).astype(np.float32)
X_test_lstm_scaled = lstm_scaler.transform(X_test).astype(np.float32)

X_train_lstm, y_train_lstm = make_lstm_sequences(
    train_df,
    X_train_lstm_scaled,
    y_train.to_numpy(),
    lstm_sequence_length,
)
X_test_lstm, y_test_lstm = make_lstm_sequences(
    test_df,
    X_test_lstm_scaled,
    y_test.to_numpy(),
    lstm_sequence_length,
)

validation_start = int(len(X_train_lstm) * 0.8)
X_lstm_fit = X_train_lstm[:validation_start]
y_lstm_fit = y_train_lstm[:validation_start]
X_lstm_val = X_train_lstm[validation_start:]
y_lstm_val = y_train_lstm[validation_start:]

train_dataset = TensorDataset(
    torch.tensor(X_lstm_fit, dtype=torch.float32),
    torch.tensor(y_lstm_fit, dtype=torch.long),
)
validation_dataset = TensorDataset(
    torch.tensor(X_lstm_val, dtype=torch.float32),
    torch.tensor(y_lstm_val, dtype=torch.long),
)
train_loader = DataLoader(train_dataset, batch_size=lstm_batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=lstm_batch_size, shuffle=False)

lstm_layer = nn.LSTM(
    input_size=len(selected_features),
    hidden_size=lstm_hidden_size,
    batch_first=True,
)
lstm_output_layer = nn.Linear(lstm_hidden_size, 3)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    list(lstm_layer.parameters()) + list(lstm_output_layer.parameters()),
    lr=lstm_learning_rate,
)

best_validation_loss = np.inf
best_lstm_state = None
best_output_state = None
epochs_without_improvement = 0

for epoch in range(lstm_epochs):
    lstm_layer.train()
    lstm_output_layer.train()

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        lstm_output, _ = lstm_layer(batch_X)
        logits = lstm_output_layer(lstm_output[:, -1, :])
        loss = loss_function(logits, batch_y)
        loss.backward()
        optimizer.step()

    lstm_layer.eval()
    lstm_output_layer.eval()
    validation_losses = []

    with torch.no_grad():
        for batch_X, batch_y in validation_loader:
            lstm_output, _ = lstm_layer(batch_X)
            logits = lstm_output_layer(lstm_output[:, -1, :])
            validation_loss = loss_function(logits, batch_y)
            validation_losses.append(validation_loss.item())

    mean_validation_loss = float(np.mean(validation_losses))

    if mean_validation_loss < best_validation_loss:
        best_validation_loss = mean_validation_loss
        best_lstm_state = {key: value.detach().clone() for key, value in lstm_layer.state_dict().items()}
        best_output_state = {key: value.detach().clone() for key, value in lstm_output_layer.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= lstm_patience:
        break

lstm_layer.load_state_dict(best_lstm_state)
lstm_output_layer.load_state_dict(best_output_state)

lstm_layer.eval()
lstm_output_layer.eval()

lstm_predictions = []
test_dataset = TensorDataset(torch.tensor(X_test_lstm, dtype=torch.float32))
test_loader = DataLoader(test_dataset, batch_size=lstm_batch_size, shuffle=False)

with torch.no_grad():
    for (batch_X,) in test_loader:
        lstm_output, _ = lstm_layer(batch_X)
        logits = lstm_output_layer(lstm_output[:, -1, :])
        batch_pred = torch.argmax(logits, dim=1).numpy()
        lstm_predictions.append(batch_pred)

lstm_pred = np.concatenate(lstm_predictions)

lstm_cm = pd.DataFrame(
    confusion_matrix(y_test_lstm, lstm_pred, labels=[0, 1, 2]),
    index=class_names,
    columns=class_names,
)
display(lstm_cm)

print(classification_report(y_test_lstm, lstm_pred, labels=[0, 1, 2], target_names=class_names, zero_division=0))

lstm_report = classification_report(y_test_lstm, lstm_pred, labels=[0, 1, 2], target_names=class_names, zero_division=0, output_dict=True)
results.append({
    "model": "LSTM",
    "accuracy": lstm_report["accuracy"],
    "macro_f1": lstm_report["macro avg"]["f1-score"],
    "weighted_f1": lstm_report["weighted avg"]["f1-score"],
})


,down,up,neutral
down,13571,13682,11596
up,13479,14749,10633
neutral,12474,11083,15372


              precision    recall  f1-score   support

        down       0.34      0.35      0.35     38849
          up       0.37      0.38      0.38     38861
     neutral       0.41      0.39      0.40     38929

    accuracy                           0.37    116639
   macro avg       0.38      0.37      0.37    116639
weighted avg       0.38      0.37      0.37    116639



# Final Comparison

In [19]:
final_comparison = pd.DataFrame(results)
display(final_comparison)


,model,accuracy,macro_f1,weighted_f1
0,Random Baseline,0.333605,0.333604,0.333604
1,Logistic Regression,0.382322,0.371893,0.371928
2,Random Forest,0.376960,0.375361,0.375379
3,XGBoost,0.368247,0.368352,0.368354
4,LSTM,0.374592,0.374804,0.374823


In [20]:
def show_feature_importance(top_n=20):
    logistic_importance = np.abs(logistic_model.coef_).mean(axis=0)
    rf_importance = rf_model.feature_importances_

    xgb_gain = xgb_model.get_booster().get_score(importance_type="gain")
    xgb_importance = np.array([
        xgb_gain.get(feature, xgb_gain.get(f"f{i}", 0.0))
        for i, feature in enumerate(selected_features)
    ])

    importance = pd.DataFrame({
        "feature": selected_features,
        "logistic_abs_coefficient": logistic_importance,
        "random_forest_importance": rf_importance,
        "xgboost_gain": xgb_importance,
    })

    for col in ["logistic_abs_coefficient", "random_forest_importance", "xgboost_gain"]:
        importance[col + "_normalized"] = importance[col] / importance[col].sum()

    importance["average_importance"] = importance[
        [
            "logistic_abs_coefficient_normalized",
            "random_forest_importance_normalized",
            "xgboost_gain_normalized",
        ]
    ].mean(axis=1)

    importance = importance.sort_values("average_importance", ascending=False).reset_index(drop=True)
    display(importance.head(top_n))
    return importance

feature_importance = show_feature_importance(top_n=20)


,feature,logistic_abs_coefficient,random_forest_importance,xgboost_gain,logistic_abs_coefficient_normalized,random_forest_importance_normalized,xgboost_gain_normalized,average_importance
0,event_count_20,0.115294,0.156839,32.741322,0.181076,0.156839,0.088922,0.142279
1,realized_vol_20,0.065025,0.129789,24.901892,0.102126,0.129789,0.067631,0.099849
2,event_count_10,0.029015,0.105132,11.308847,0.045570,0.105132,0.030714,0.060472
3,realized_vol_10,0.017498,0.082866,11.548572,0.027482,0.082866,0.031365,0.047237
4,spread_ticks,0.042894,0.033254,13.867048,0.067367,0.033254,0.037662,0.046094
5,trade_count_20,0.048714,0.038827,8.139658,0.076508,0.038827,0.022107,0.045814
6,bb_width_20,0.014456,0.049406,9.555449,0.022705,0.049406,0.025952,0.032687
7,trade_count_10,0.026089,0.017711,6.911336,0.040974,0.017711,0.018771,0.025818
8,macd,0.020647,0.019094,8.909064,0.032427,0.019094,0.024196,0.025239
9,sma_20_dist,0.021036,0.015365,8.314073,0.033039,0.015365,0.022580,0.023662
